# PySpark drills 8: nested and semi-structured data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-08-nested-and-semi-structured-data.ipynb)

Practice for [PySpark 8: nested and semi-structured data](pyspark-08-nested-and-semi-structured-data.ipynb). Each exercise uses a small nested input created in its data cell, or `order_items`.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

## Exercise 1: Explode tags without losing events

**Difficulty:** Easy · about 5 min

Return one row per event and tag, with `event_id` and `tag`, keeping events whose tag list is null or empty (with a
null tag). Order by `event_id` then `tag`.

**Sample input**

**`tagged`**

| event_id | tags |
| --- | --- |
| e1 | ['sale', 'new'] |
| e2 | [] |
| e3 | null |

**Expected output**

| event_id | tag |
| --- | --- |
| e1 | new |
| e1 | sale |
| e2 | null |
| e3 | null |

In [ ]:
# Sample data for this exercise.
tagged = spark.createDataFrame([("e1", ["sale", "new"]), ("e2", []), ("e3", None)], "event_id string, tags array<string>")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = tagged.select("event_id", F.explode_outer("tags").alias("tag")).orderBy("event_id", "tag")
result.show()

## Exercise 2: SKUs per order as a sorted list

**Difficulty:** Medium · about 10 min

Return one row per order with `order_id`, `skus` (the order's SKUs as an array, sorted) and `lines` (the number of
lines), ordered by `order_id`.

**Expected output**

| order_id | skus | lines |
| --- | --- | --- |
| 1 | ['SKU-4001', 'SKU-4007'] | 2 |
| 2 | ['SKU-4002', 'SKU-4009'] | 2 |
| 3 | ['SKU-4002'] | 1 |
| 4 | ['SKU-4003', 'SKU-4004'] | 2 |
| 5 | ['SKU-4005'] | 1 |
| 6 | ['SKU-4003'] | 1 |
| 7 | ['SKU-4006', 'SKU-4008'] | 2 |
| 8 | ['SKU-4010'] | 1 |
| 9 | ['SKU-4011'] | 1 |
| 10 | ['SKU-4012'] | 1 |
| 11 | ['SKU-4013'] | 1 |
| 12 | ['SKU-4005', 'SKU-4014'] | 2 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    order_items.groupBy("order_id")
    .agg(F.sort_array(F.collect_list("sku")).alias("skus"), F.count("*").alias("lines"))
    .orderBy("order_id")
)
result.show(truncate=False)

## Exercise 3: Parse webhook payloads

**Difficulty:** Medium · about 15 min

Parse `payload` with the schema `customer string, items array<struct<sku:string, qty:int>>`. Return `event_id`,
`customer`, `units` (the sum of `qty`, 0 when items are missing or empty) and `is_valid` (false when the payload is
not JSON). Order by `event_id`.

**Sample input**

**`payloads`**

| event_id | payload |
| --- | --- |
| w1 | {"customer": "c1", "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]} |
| w2 | {"customer": "c2", "items": []} |
| w3 | {"customer": "c3"} |
| w4 | not json |

**Expected output**

| event_id | customer | units | is_valid |
| --- | --- | --- | --- |
| w1 | c1 | 3 | true |
| w2 | c2 | 0 | true |
| w3 | c3 | 0 | true |
| w4 | null | 0 | false |

In [ ]:
# Sample data for this exercise.
payloads = spark.createDataFrame(
    [("w1", '{"customer": "c1", "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}'),
     ("w2", '{"customer": "c2", "items": []}'), ("w3", '{"customer": "c3"}'), ("w4", "not json")],
    "event_id string, payload string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
parsed = payloads.withColumn("data", F.from_json("payload", "customer string, items array<struct<sku:string, qty:int>>"))
result = parsed.select(
    "event_id",
    F.col("data.customer").alias("customer"),
    F.coalesce(F.aggregate("data.items", F.lit(0), lambda total, item: total + item["qty"]), F.lit(0)).alias("units"),
    F.col("data.customer").isNotNull().alias("is_valid"),
).orderBy("event_id")
result.show()

## Exercise 4: Order totals without exploding

**Difficulty:** Medium · about 15 min

Each order holds its lines as an array of structs. Using higher-order functions only (no `explode`), return
`order_id`, `total` (sum of `qty * price`) and `has_bulk_line` (true if any line has `qty` of 10 or more), ordered by
`order_id`.

**Expected output**

| order_id | total | has_bulk_line |
| --- | --- | --- |
| 1 | 120.0 | false |
| 2 | 1180.0 | true |
| 3 | 0.0 | false |

In [ ]:
# Sample data for this exercise.
nested = spark.createDataFrame(
    [(1, [("A", 2, 12.5), ("B", 1, 95.0)]), (2, [("C", 20, 59.0)]), (3, [])],
    "order_id int, lines array<struct<sku:string, qty:int, price:double>>",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = nested.select(
    "order_id",
    F.aggregate("lines", F.lit(0.0), lambda total, line: total + line["qty"] * line["price"]).alias("total"),
    F.exists("lines", lambda line: line["qty"] >= 10).alias("has_bulk_line"),
).orderBy("order_id")
result.show()

## Exercise 5: Read a value from a map with a default

**Difficulty:** Easy · about 5 min

Each product has a map of attributes. Return `sku` and `color`, using `unspecified` when the map has no `color` key.
Order by `sku`.

**Expected output**

| sku | color |
| --- | --- |
| SKU-1 | red |
| SKU-2 | unspecified |
| SKU-3 | unspecified |

In [ ]:
# Sample data for this exercise.
attributes = spark.createDataFrame(
    [("SKU-1", {"color": "red", "size": "M"}), ("SKU-2", {"size": "L"}), ("SKU-3", {})], "sku string, attrs map<string,string>"
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = attributes.select("sku", F.coalesce(F.col("attrs")["color"], F.lit("unspecified")).alias("color")).orderBy("sku")
result.show()

**Next:** [PySpark 9: Python UDFs and pandas](pyspark-09-python-udfs-and-pandas.ipynb)